In [ ]:
%cd /content/paper-theater-ai

from src.config import Paths, PipelineConfig
from src.io_utils import load_mask
from src.vectorize import mask_to_polygons
from src.fabrication import merge_polygons, thicken_fragile_parts, remove_tiny_parts
from src.export_svg import save_svg
import numpy as np

In [ ]:
def export_branch(branch_results, branch_name, image_shape):
    paths = Paths()
    cfg = PipelineConfig()

    h, w = image_shape[:2]

    for layer_idx in range(cfg.target_num_layers):
        masks = []
        for item in branch_results:
            if item["layer"] == layer_idx:
                masks.append(load_mask(item["mask_path"]))

        if not masks:
            continue

        merged = np.zeros_like(masks[0], dtype=np.uint8)
        for m in masks:
            merged = ((merged > 0) | (m > 0)).astype(np.uint8)

        polys = mask_to_polygons(merged, cfg.simplify_tolerance)
        geom = merge_polygons(polys)
        geom = remove_tiny_parts(geom, 100)
        geom = thicken_fragile_parts(geom, 2)

        out_path = paths.layers_svg_dir / f"{branch_name}_layer_{layer_idx}.svg"
        save_svg(geom, out_path, w, h)
        print("saved", out_path)

In [ ]:
from PIL import Image
from pathlib import Path
import json

image_path = Paths().input_dir / "scene.png"
img = Image.open(image_path).convert("RGB")
img_np = np.array(img)

# reuse branch results from earlier notebook or load from saved json if you stored them
export_branch(heuristic_results, "heuristic", img_np.shape)
export_branch(amodal_results, "amodal", img_np.shape)
export_branch(openai_results, "openai", img_np.shape)

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
from PIL import Image

paths = Paths()

for p in sorted(paths.layers_svg_dir.glob("*.svg")):
    print(p.name)